
# Song Genre NLP Classifier — Full Upgrade Notebook

This notebook implements all of the upgrades discussed for **multilabel song-genre prediction from lyrics**:

1. **Better multilabel splitting** with iterative stratification when available
2. **CNN text model** (`Embedding + Conv1D + GlobalMaxPooling1D`)
3. **BiLSTM text model**
4. **Hybrid text + audio model**
5. **Label cleanup / broader-genre comparison**

It uses your CSV directly and keeps the workflow runnable end to end.


In [ ]:

# Optional: install iterative-stratification if it is not already available.
# Uncomment if needed in a fresh environment.
# %pip install iterative-stratification


In [ ]:

import os
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, hamming_loss, precision_score, recall_score
from sklearn.model_selection import train_test_split

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except Exception:
    HAS_ITERSTRAT = False

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout,
    BatchNormalization, Bidirectional, LSTM, SpatialDropout1D, Concatenate
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.metrics import Precision, Recall

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


print('Iterative stratification available:', HAS_ITERSTRAT)


In [ ]:

# Resolve CSV path automatically.
CANDIDATE_PATHS = [
    '/mnt/data/train_with_lyrics(3).csv',
    './train_with_lyrics(3).csv',
    './train_with_lyrics.csv',
    '/Users/tukermoose/Downloads/genius_docker_project/data/train_with_lyrics.csv',
]

CSV_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if CSV_PATH is None:
    raise FileNotFoundError(f'Could not find the CSV. Checked: {CANDIDATE_PATHS}')

print('Using CSV:', CSV_PATH)
df = pd.read_csv(CSV_PATH)
print('Raw shape:', df.shape)
df.head(2)


## 1) Clean lyrics, merge duplicate songs, and create fine + broad labels

In [ ]:
def clean_lyrics(text: str) -> str:
    if not isinstance(text, str):
        return ''

    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\[.*?\]', ' ', text)
    text = re.sub(r"[^a-zA-Z0-9'\s]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text


def first_nonempty(series):
    for value in series:
        if isinstance(value, str) and value.strip():
            return value
    return ''


def broad_bucket(label: str) -> str:
    g = str(label).lower().strip()
    rules = [
        ('metal', ['metal', 'black-metal', 'death-metal', 'heavy-metal', 'grindcore', 'hardcore']),
        ('rock', ['rock', 'alt-rock', 'alternative', 'indie-rock', 'psych-rock', 'rock-n-roll', 'garage', 'grunge']),
        ('punk_emo', ['punk', 'emo', 'ska-punk', 'post-punk']),
        ('pop', ['pop', 'power-pop', 'synth-pop', 'dance-pop', 'electropop', 'indie-pop']),
        ('hiphop_rap', ['hip-hop', 'rap', 'trap', 'drill']),
        ('rnb_soul', ['r-n-b', 'r&b', 'soul', 'funk', 'motown', 'neo-soul']),
        ('electronic_dance', ['edm', 'electro', 'house', 'techno', 'trance', 'dubstep', 'drum-and-bass', 'dance', 'disco', 'club', 'minimal-techno']),
        ('jazz_blues', ['jazz', 'blues', 'swing', 'bebop']),
        ('country_folk', ['country', 'folk', 'bluegrass', 'americana']),
        ('latin', ['latin', 'reggaeton', 'salsa', 'tango', 'samba', 'forro', 'pagode', 'sertanejo', 'spanish']),
        ('reggae_ska_dub', ['reggae', 'ska', 'dub']),
        ('classical_opera', ['classical', 'opera', 'piano', 'violin', 'orchestra', 'romance']),
        ('ambient_newage', ['ambient', 'new-age', 'sleep', 'study', 'lo-fi', 'chill']),
        ('world_traditional', ['afro', 'turkish', 'indian', 'world', 'iranian', 'malay', 'german', 'french']),
        ('k_j_anime', ['k-pop', 'j-pop', 'j-rock', 'anime']),
        ('gospel_christian', ['gospel', 'christian']),
        ('kids_comedy', ['children', 'kids', 'comedy']),
    ]
    for bucket, keywords in rules:
        if any(k in g for k in keywords):
            return bucket
    return 'other'


df['lyrics_clean'] = df['lyrics'].apply(clean_lyrics)

song_df = (
    df.groupby('track_id', as_index=False)
      .agg({
          'track_name': 'first',
          'artists': 'first',
          'album_name': 'first',
          'lyrics_clean': first_nonempty,
          'track_genre': lambda s: sorted(set(map(str, s.dropna()))),
          'danceability': 'first',
          'energy': 'first',
          'key': 'first',
          'loudness': 'first',
          'mode': 'first',
          'speechiness': 'first',
          'acousticness': 'first',
          'instrumentalness': 'first',
          'liveness': 'first',
          'valence': 'first',
          'tempo': 'first',
          'time_signature': 'first',
          'duration_ms': 'first',
          'explicit': 'first',
          'popularity': 'first',
      })
      .rename(columns={'lyrics_clean': 'lyrics', 'track_genre': 'genres_fine'})
)

song_df['genres_broad'] = song_df['genres_fine'].apply(lambda gs: sorted(set(broad_bucket(g) for g in gs)))
song_df['lyrics_word_count'] = song_df['lyrics'].str.split().apply(len)

MIN_WORDS = 20
song_df = song_df[song_df['lyrics'].str.strip().ne('') & (song_df['lyrics_word_count'] >= MIN_WORDS)].copy()
song_df.reset_index(drop=True, inplace=True)

print('Merged song-level shape:', song_df.shape)
print('Average fine labels per song:', song_df['genres_fine'].apply(len).mean().round(2))
print('Average broad labels per song:', song_df['genres_broad'].apply(len).mean().round(2))
song_df[['track_name', 'artists', 'genres_fine', 'genres_broad']].head(5)

In [ ]:

print('Unique fine-grained genres:', len(sorted({g for labels in song_df['genres_fine'] for g in labels})))
print('Broad buckets:', sorted({g for labels in song_df['genres_broad'] for g in labels}))


## 2) Build fine-grained and broad multilabel targets

In [ ]:

mlb_fine = MultiLabelBinarizer()
Y_fine = mlb_fine.fit_transform(song_df['genres_fine'])

mlb_broad = MultiLabelBinarizer()
Y_broad = mlb_broad.fit_transform(song_df['genres_broad'])

print('Fine target shape:', Y_fine.shape)
print('Broad target shape:', Y_broad.shape)


## 3) Better multilabel splitting (iterative stratification when available)

In [ ]:
X_text = song_df['lyrics'].astype(str).values
meta_titles = song_df['track_name'].values
meta_artists = song_df['artists'].values

AUDIO_FEATURES = [
    'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
    'time_signature', 'duration_ms', 'explicit', 'popularity'
]

X_audio = song_df[AUDIO_FEATURES].copy()
X_audio['explicit'] = pd.to_numeric(X_audio['explicit'], errors='coerce')
X_audio = X_audio.fillna(X_audio.median(numeric_only=True))
X_audio = X_audio.astype(float).values


def multilabel_split(X_text, X_audio, Y, seed=SEED, test_size=0.15, val_size=0.15):
    n = len(X_text)
    indices = np.arange(n)

    if HAS_ITERSTRAT:
        msss1 = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed
        )
        train_val_idx, test_idx = next(msss1.split(indices.reshape(-1, 1), Y))

        Y_train_val = Y[train_val_idx]
        relative_val_size = val_size / (1 - test_size)

        msss2 = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=relative_val_size,
            random_state=seed
        )
        train_idx_rel, val_idx_rel = next(
            msss2.split(train_val_idx.reshape(-1, 1), Y_train_val)
        )

        train_idx = train_val_idx[train_idx_rel]
        val_idx = train_val_idx[val_idx_rel]
        split_name = 'iterative stratification'
    else:
        relative_val_size = val_size / (1 - test_size)

        train_val_idx, test_idx = train_test_split(
            indices,
            test_size=test_size,
            random_state=seed,
            shuffle=True
        )
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=relative_val_size,
            random_state=seed,
            shuffle=True
        )
        split_name = 'random split fallback (iterative-stratification not installed)'

    return {
        'method': split_name,
        'train_idx': train_idx,
        'val_idx': val_idx,
        'test_idx': test_idx,
        'X_train_text': X_text[train_idx],
        'X_val_text': X_text[val_idx],
        'X_test_text': X_text[test_idx],
        'X_train_audio': X_audio[train_idx],
        'X_val_audio': X_audio[val_idx],
        'X_test_audio': X_audio[test_idx],
        'Y_train': Y[train_idx],
        'Y_val': Y[val_idx],
        'Y_test': Y[test_idx],
        'titles_test': meta_titles[test_idx],
        'artists_test': meta_artists[test_idx],
    }

split_fine = multilabel_split(X_text, X_audio, Y_fine)
split_broad = multilabel_split(X_text, X_audio, Y_broad)

print('Fine split method:', split_fine['method'])
print('Fine sizes:', split_fine['Y_train'].shape, split_fine['Y_val'].shape, split_fine['Y_test'].shape)
print('Broad split method:', split_broad['method'])
print('Broad sizes:', split_broad['Y_train'].shape, split_broad['Y_val'].shape, split_broad['Y_test'].shape)

## 4) Shared evaluation helpers

In [ ]:

def evaluate_multilabel(y_true, y_probs, threshold=0.5, model_name='Model'):
    y_pred = (y_probs >= threshold).astype(int)
    return {
        'model': model_name,
        'threshold': float(threshold),
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'pred_binary': y_pred,
    }


def tune_threshold(y_true, y_probs, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.1, 0.71, 0.05)
    rows = []
    for t in thresholds:
        ev = evaluate_multilabel(y_true, y_probs, threshold=t, model_name='tmp')
        rows.append({
            'threshold': t,
            'micro_f1': ev['micro_f1'],
            'macro_f1': ev['macro_f1'],
            'hamming_loss': ev['hamming_loss'],
        })
    return pd.DataFrame(rows).sort_values('micro_f1', ascending=False)


def show_top_labels(binary_row, classes):
    return [classes[i] for i, v in enumerate(binary_row) if v == 1]


## 5) Fine-grained baseline — TF-IDF + One-vs-Rest Logistic Regression

In [ ]:

X_train_text = split_fine['X_train_text']
X_val_text = split_fine['X_val_text']
X_test_text = split_fine['X_test_text']
Y_train = split_fine['Y_train']
Y_val = split_fine['Y_val']
Y_test = split_fine['Y_test']

MAX_FEATURES_TFIDF = 30000

tfidf = TfidfVectorizer(
    max_features=MAX_FEATURES_TFIDF,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)
X_test_tfidf = tfidf.transform(X_test_text)

baseline = OneVsRestClassifier(LogisticRegression(max_iter=2000, C=3.0))
baseline.fit(X_train_tfidf, Y_train)

val_probs_baseline = baseline.predict_proba(X_val_tfidf)
test_probs_baseline = baseline.predict_proba(X_test_tfidf)

baseline_threshold_results = tune_threshold(Y_val, val_probs_baseline)
BEST_BASELINE_THRESHOLD = float(baseline_threshold_results.iloc[0]['threshold'])

baseline_eval = evaluate_multilabel(
    Y_test, test_probs_baseline,
    threshold=BEST_BASELINE_THRESHOLD,
    model_name='Fine TF-IDF + OvR Logistic Regression'
)

pd.DataFrame([{k: v for k, v in baseline_eval.items() if k != 'pred_binary'}])


## 6) Prepare tokenizer + padded sequences for neural text models

In [ ]:

VOCAB_SIZE = 30000
MAX_LEN = 400
EMBED_DIM = 128
EPOCHS = 8
BATCH_SIZE = 32


def make_tokenized_text(train_text, val_text, test_text, vocab_size=VOCAB_SIZE, max_len=MAX_LEN):
    tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
    tokenizer.fit_on_texts(train_text)

    X_train_seq = tokenizer.texts_to_sequences(train_text)
    X_val_seq = tokenizer.texts_to_sequences(val_text)
    X_test_seq = tokenizer.texts_to_sequences(test_text)

    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
    X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post', truncating='post')
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')
    return tokenizer, X_train_pad, X_val_pad, X_test_pad


tokenizer_fine, X_train_pad, X_val_pad, X_test_pad = make_tokenized_text(X_train_text, X_val_text, X_test_text)
print(X_train_pad.shape, X_val_pad.shape, X_test_pad.shape)


In [ ]:

def make_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-6),
    ]


def build_cnn_model(num_labels, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=max_len),
        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(256, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.5),
        Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.4),
        Dense(num_labels, activation='sigmoid'),
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[Precision(name='precision'), Recall(name='recall')],
    )
    return model


def build_bilstm_model(num_labels, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=max_len),
        SpatialDropout1D(0.2),
        Bidirectional(LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.0)),
        Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_labels, activation='sigmoid'),
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[Precision(name='precision'), Recall(name='recall')],
    )
    return model


def build_hybrid_model(num_labels, num_audio_features, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN):
    text_input = Input(shape=(max_len,), name='text_input')
    x = Embedding(vocab_size, embed_dim, input_length=max_len)(text_input)
    x = Conv1D(128, 5, activation='relu')(x)
    x = GlobalMaxPooling1D()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.4)(x)

    audio_input = Input(shape=(num_audio_features,), name='audio_input')
    a = Dense(64, activation='relu')(audio_input)
    a = BatchNormalization()(a)
    a = Dropout(0.3)(a)

    combined = Concatenate()([x, a])
    combined = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(combined)
    combined = BatchNormalization()(combined)
    combined = Dropout(0.4)(combined)
    output = Dense(num_labels, activation='sigmoid')(combined)

    model = Model(inputs=[text_input, audio_input], outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[Precision(name='precision'), Recall(name='recall')],
    )
    return model


## 7) Fine-grained CNN text model

In [ ]:

cnn_model = build_cnn_model(num_labels=Y_train.shape[1])
cnn_model.summary()


In [ ]:

history_cnn = cnn_model.fit(
    X_train_pad, Y_train,
    validation_data=(X_val_pad, Y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1,
)

val_probs_cnn = cnn_model.predict(X_val_pad, verbose=0)
test_probs_cnn = cnn_model.predict(X_test_pad, verbose=0)
cnn_threshold_results = tune_threshold(Y_val, val_probs_cnn)
BEST_CNN_THRESHOLD = float(cnn_threshold_results.iloc[0]['threshold'])
cnn_eval = evaluate_multilabel(
    Y_test, test_probs_cnn,
    threshold=BEST_CNN_THRESHOLD,
    model_name='Fine CNN text model'
)
pd.DataFrame([{k: v for k, v in cnn_eval.items() if k != 'pred_binary'}])


## 8) Fine-grained BiLSTM text model

In [ ]:

bilstm_model = build_bilstm_model(num_labels=Y_train.shape[1])
bilstm_model.summary()


In [ ]:

history_bilstm = bilstm_model.fit(
    X_train_pad, Y_train,
    validation_data=(X_val_pad, Y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1,
)

val_probs_bilstm = bilstm_model.predict(X_val_pad, verbose=0)
test_probs_bilstm = bilstm_model.predict(X_test_pad, verbose=0)
bilstm_threshold_results = tune_threshold(Y_val, val_probs_bilstm)
BEST_BILSTM_THRESHOLD = float(bilstm_threshold_results.iloc[0]['threshold'])
bilstm_eval = evaluate_multilabel(
    Y_test, test_probs_bilstm,
    threshold=BEST_BILSTM_THRESHOLD,
    model_name='Fine BiLSTM text model'
)
pd.DataFrame([{k: v for k, v in bilstm_eval.items() if k != 'pred_binary'}])


## 9) Fine-grained hybrid model — lyrics + audio features

In [ ]:

X_train_audio = split_fine['X_train_audio']
X_val_audio = split_fine['X_val_audio']
X_test_audio = split_fine['X_test_audio']

scaler = StandardScaler()
X_train_audio_scaled = scaler.fit_transform(X_train_audio)
X_val_audio_scaled = scaler.transform(X_val_audio)
X_test_audio_scaled = scaler.transform(X_test_audio)

hybrid_model = build_hybrid_model(
    num_labels=Y_train.shape[1],
    num_audio_features=X_train_audio_scaled.shape[1]
)
hybrid_model.summary()


In [ ]:

history_hybrid = hybrid_model.fit(
    {'text_input': X_train_pad, 'audio_input': X_train_audio_scaled},
    Y_train,
    validation_data=({'text_input': X_val_pad, 'audio_input': X_val_audio_scaled}, Y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1,
)

val_probs_hybrid = hybrid_model.predict({'text_input': X_val_pad, 'audio_input': X_val_audio_scaled}, verbose=0)
test_probs_hybrid = hybrid_model.predict({'text_input': X_test_pad, 'audio_input': X_test_audio_scaled}, verbose=0)
hybrid_threshold_results = tune_threshold(Y_val, val_probs_hybrid)
BEST_HYBRID_THRESHOLD = float(hybrid_threshold_results.iloc[0]['threshold'])
hybrid_eval = evaluate_multilabel(
    Y_test, test_probs_hybrid,
    threshold=BEST_HYBRID_THRESHOLD,
    model_name='Fine hybrid text + audio model'
)
pd.DataFrame([{k: v for k, v in hybrid_eval.items() if k != 'pred_binary'}])


## 10) Broader-genre comparison with cleaned labels

In [ ]:

X_train_text_b = split_broad['X_train_text']
X_val_text_b = split_broad['X_val_text']
X_test_text_b = split_broad['X_test_text']
Y_train_b = split_broad['Y_train']
Y_val_b = split_broad['Y_val']
Y_test_b = split_broad['Y_test']

# Fast broad-label baseline
b_tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    sublinear_tf=True,
)
X_train_b_tfidf = b_tfidf.fit_transform(X_train_text_b)
X_val_b_tfidf = b_tfidf.transform(X_val_text_b)
X_test_b_tfidf = b_tfidf.transform(X_test_text_b)

b_baseline = OneVsRestClassifier(LogisticRegression(max_iter=2000, C=3.0))
b_baseline.fit(X_train_b_tfidf, Y_train_b)

val_probs_b = b_baseline.predict_proba(X_val_b_tfidf)
test_probs_b = b_baseline.predict_proba(X_test_b_tfidf)
threshold_results_b = tune_threshold(Y_val_b, val_probs_b)
BEST_BROAD_THRESHOLD = float(threshold_results_b.iloc[0]['threshold'])

broad_eval = evaluate_multilabel(
    Y_test_b, test_probs_b,
    threshold=BEST_BROAD_THRESHOLD,
    model_name='Broad-label TF-IDF + OvR Logistic Regression'
)

pd.DataFrame([{k: v for k, v in broad_eval.items() if k != 'pred_binary'}])


## 11) Compare models

In [ ]:

comparison = pd.DataFrame([
    {k: v for k, v in baseline_eval.items() if k != 'pred_binary'},
    {k: v for k, v in cnn_eval.items() if k != 'pred_binary'},
    {k: v for k, v in bilstm_eval.items() if k != 'pred_binary'},
    {k: v for k, v in hybrid_eval.items() if k != 'pred_binary'},
    {k: v for k, v in broad_eval.items() if k != 'pred_binary'},
]).sort_values(['micro_f1', 'macro_f1'], ascending=False)
comparison


## 12) Inspect sample fine-grained predictions

In [ ]:

sample_probs = test_probs_hybrid
sample_threshold = BEST_HYBRID_THRESHOLD
sample_pred = (sample_probs >= sample_threshold).astype(int)

examples = []
for i in range(min(10, len(sample_pred))):
    examples.append({
        'track_name': split_fine['titles_test'][i],
        'artists': split_fine['artists_test'][i],
        'true_labels': show_top_labels(Y_test[i], mlb_fine.classes_),
        'pred_labels': show_top_labels(sample_pred[i], mlb_fine.classes_),
    })

pd.DataFrame(examples)



## 13) Notes

- The **fine-grained task** keeps the original multilabel structure.
- The **broad-label task** tests whether label cleanup makes the problem easier and more interpretable.
- The **hybrid model** is usually the most promising practical direction because genre is encoded in both lyrics and sound-related features.
- If training time is too long, lower `EPOCHS`, lower `VOCAB_SIZE`, or shorten `MAX_LEN`.
